# Dynamika pohybu vozidla

Dynamika sa zaoberá príčinami pohybu, teda silami ktoré pohyb
spôsobujú alebo brzdia. V tomto notebooku aplikujeme Newtonove
zákony na reálne vozidlo a skúmame vzťah medzi silou, hmotnosťou
a zrýchlením.

Pri otvorení v JupyterLab sú k dispozícii interaktívne slidery –
hodnoty môžeš meniť aj priamo v bloku PARAMETRE v kóde.

## Teoretický základ

Na pohybujúce sa vozidlo pôsobia tri hlavné sily:

**Hnacia sila motora:**
$$F_{motor} = \frac{P}{v}$$

**Aerodynamický odpor:**
$$F_{vzduch} = \frac{1}{2} \cdot \rho \cdot C_d \cdot A \cdot v^2$$

**Valivý odpor:**
$$F_{valivý} = \mu \cdot m \cdot g$$

**Výsledné zrýchlenie** podľa Newtonovho druhého zákona:
$$a = \frac{F_{motor} - F_{vzduch} - F_{valivý}}{m}$$

Vozidlo dosiahne maximálnu rýchlosť keď
$F_{motor} = F_{vzduch} + F_{valivý}$.

## Silová analýza a maximálna rýchlosť

![Silová analýza a zrýchlenie vozidla](_images/dynamika_1.png)
*Obr. Silová analýza vozidla – hnacia sila vs. odpory a výsledné zrýchlenie*

In [1]:
# ============================================================
# AKO POUŽÍVAŤ TENTO NOTEBOOK V JUPYTERLAB:
# 1. Zmeň hodnoty v bloku PARAMETRE nižšie
# 2. Spusti bunku klávesou Shift+Enter
# 3. Alebo použi slidery ktoré sa zobrazia pod bunkou
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
%matplotlib inline
plt.style.use('dark_background')

# Fyzikálne konštanty – nemeniť
RHO = 1.225   # hustota vzduchu [kg/m³]
G   = 9.81    # gravitačné zrýchlenie [m/s²]
A   = 2.2     # čelná plocha vozidla [m²]

# ============================================================
# PARAMETRE – môžeš meniť tu ALEBO cez slidery nižšie
# ============================================================
m_def    = 1400    # hmotnosť vozidla [kg]       → typické auto: 1000–2000 kg
P_kW_def = 100     # výkon motora [kW]            → malé: 60–100, športové: 200+
Cd_def   = 0.30    # koef. aerodyn. odporu [-]    → typické: 0.25–0.45
mu_def   = 0.015   # koef. valivého odporu [-]    → asfalt: 0.010–0.020
# ============================================================

out = widgets.Output()

def zobraz_dynamiku(m, P_kW, Cd, mu):
    P = P_kW * 1000   # prepočet kW → W

    # Rozsah rýchlostí pre výpočet (0.5 až 90 m/s)
    v     = np.linspace(0.5, 90, 800)
    v_kmh = v * 3.6    # prepočet m/s → km/h

    # Výpočet jednotlivých síl
    F_motor  = P / v                          # hnacia sila klesá s rýchlosťou
    F_vzduch = 0.5 * RHO * Cd * A * v**2     # aerodyn. odpor rastie kvadraticky
    F_valivý = mu * m * G * np.ones_like(v)  # valivý odpor je konštantný
    F_odpor  = F_vzduch + F_valivý            # celkový odpor
    F_celk   = F_motor - F_odpor             # výsledná sila na vozidlo
    a        = np.clip(F_celk / m, -25, 25)  # zrýchlenie (a = F/m)

    # Maximálna rýchlosť – kde sa hnacia sila rovná odporu (F_celk ≈ 0)
    idx_vmax = np.argmin(np.abs(F_celk))
    v_max    = v_kmh[idx_vmax]

    with out:
        clear_output(wait=True)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
        fig.patch.set_facecolor('#0d0d1a')

        # ── Graf síl ─────────────────────────────────────────
        ax1.set_facecolor('#0d0d1a')
        ax1.plot(v_kmh, F_motor,  color='#00BFFF', lw=2.2,
                 label='Hnacia sila F_motor')
        ax1.plot(v_kmh, F_vzduch, color='#FF4444', lw=2.2,
                 label='Aerodyn. odpor')
        ax1.plot(v_kmh, F_valivý, color='#FFA500', lw=2.0, linestyle='--',
                 label='Valivý odpor')
        ax1.plot(v_kmh, F_odpor,  color='#FF0000', lw=2.0, linestyle=':',
                 label='Celkový odpor')
        ax1.axvline(x=v_max, color='#FFD700', lw=1.8, linestyle='--',
                    alpha=0.8, label=f'v_max ≈ {v_max:.0f} km/h')
        # Modrá oblasť = vozidlo zrýchľuje (F_motor > F_odpor)
        ax1.fill_between(v_kmh, F_motor, F_odpor,
                         where=(F_motor >= F_odpor),
                         alpha=0.10, color='#00BFFF', label='Zóna zrýchlenia')
        # Červená oblasť = vozidlo spomaľuje (F_motor < F_odpor)
        ax1.fill_between(v_kmh, F_motor, F_odpor,
                         where=(F_motor < F_odpor),
                         alpha=0.10, color='#FF4444', label='Zóna spomalenia')
        ax1.set_xlabel('Rýchlosť (km/h)', color='#cccccc')
        ax1.set_ylabel('Sila (N)', color='#cccccc')
        ax1.set_title('Silová analýza vozidla', color='white', fontsize=11)
        ax1.legend(fontsize=8, loc='upper right',
                   facecolor='#1a1a2e', edgecolor='#555555')
        ax1.grid(True, alpha=0.12)
        ax1.tick_params(colors='#aaaaaa')
        ax1.set_ylim(bottom=0)
        ax1.set_xlim(left=0)

        # ── Graf zrýchlenia ───────────────────────────────────
        ax2.set_facecolor('#0d0d1a')
        ax2.plot(v_kmh, a, color='#00FF88', lw=2.5)
        ax2.axhline(y=0, color='white', lw=1, linestyle='--', alpha=0.3)
        ax2.axvline(x=v_max, color='#FFD700', lw=1.8, linestyle='--',
                    alpha=0.8, label=f'v_max ≈ {v_max:.0f} km/h')
        # Zelená oblasť = kladné zrýchlenie
        ax2.fill_between(v_kmh, a, 0, where=(a >= 0),
                         alpha=0.15, color='#00FF88', label='Kladné zrýchlenie')
        # Červená oblasť = záporné zrýchlenie
        ax2.fill_between(v_kmh, a, 0, where=(a < 0),
                         alpha=0.15, color='#FF4444', label='Záporné zrýchlenie')
        ax2.set_xlabel('Rýchlosť (km/h)', color='#cccccc')
        ax2.set_ylabel('Zrýchlenie (m/s²)', color='#cccccc')
        ax2.set_title('Zrýchlenie vozidla', color='white', fontsize=11)
        ax2.legend(fontsize=8, facecolor='#1a1a2e', edgecolor='#555555')
        ax2.grid(True, alpha=0.12)
        ax2.tick_params(colors='#aaaaaa')
        ax2.set_xlim(left=0)

        fig.suptitle(
            f'Dynamika vozidla  │  m = {m} kg  │  P = {P_kW} kW  │  '
            f'Cd = {Cd}  │  v_max ≈ {v_max:.0f} km/h',
            fontsize=12, color='white'
        )
        plt.tight_layout()
        plt.show()

        # Výpis vypočítaných hodnôt
        print(f'Hmotnosť:            {m} kg')
        print(f'Výkon motora:        {P_kW} kW')
        print(f'Koef. odporu Cd:     {Cd}')
        print(f'Valivý odpor μ:      {mu}')
        print(f'Maximálna rýchlosť:  ≈ {v_max:.0f} km/h')

_ = widgets.interact(
    zobraz_dynamiku,
    m=widgets.IntSlider(
        value=m_def, min=500, max=3000, step=50,
        description='Hmotnosť (kg):',
        style={'description_width': '150px'},
        layout=widgets.Layout(width='500px')),
    P_kW=widgets.IntSlider(
        value=P_kW_def, min=30, max=400, step=10,
        description='Výkon (kW):',
        style={'description_width': '150px'},
        layout=widgets.Layout(width='500px')),
    Cd=widgets.FloatSlider(
        value=Cd_def, min=0.15, max=0.60, step=0.01,
        description='Cd (odpor vzduchu):',
        style={'description_width': '150px'},
        layout=widgets.Layout(width='500px')),
    mu=widgets.FloatSlider(
        value=mu_def, min=0.005, max=0.040, step=0.001,
        description='μ (valivý odpor):',
        style={'description_width': '150px'},
        layout=widgets.Layout(width='500px'))
)
display(out)

interactive(children=(IntSlider(value=1400, description='Hmotnosť (kg):', layout=Layout(width='500px'), max=30…

Output()

## Záver

- Hnacia sila klesá s rastúcou rýchlosťou, aerodynamický odpor
  naopak rastie kvadraticky s rýchlosťou
- Maximálna rýchlosť nastane keď sa hnacia sila a celkový odpor
  vyrovnajú
- Ťažšie vozidlo dosahuje nižšie zrýchlenie pri rovnakom výkone
  motora
- Aerodynamický odpor pri vysokých rýchlostiach dominuje nad
  ostatnými odpormi

Pri otvorení v JupyterLab sú k dispozícii interaktívne slidery –
parametre môžeš meniť aj priamo v bloku PARAMETRE.